# One-vs-Rest Logistic Regression with Thresholding

This notebook builds the threshold-based one-vs-rest (OVR) logistic regression competitor to the cosine-similarity threshold model.

## Plan
- Load labeled Accountancy job data and the corresponding train/validation embeddings.
- Build a multilabel target matrix where each skill is treated as its own binary classification problem.
- Train one logistic regression classifier per skill using the entity embeddings as input features.
- Convert predicted probabilities into skill labels using thresholding.
- Compare two thresholding strategies:
  - a single global probability threshold shared across all skills
  - skill-specific probability thresholds
- Evaluate with the same multilabel metrics used in the cosine-threshold notebook: micro/macro precision, recall, and F1.

## Data Used
- Labels: `../data/acc/audit_tax_accounting_jobs.csv`
- Skill taxonomy / prediction targets: `../embedding/acc_skills_embeddings.jsonl`
- Training entity embeddings: `../embedding/jobs_train_embeddings.jsonl`
- Validation entity embeddings: `../embedding/jobs_val_embeddings.jsonl`

This notebook focuses on the threshold version of OVR logistic regression, so the prediction rule is:

`P(skill_j = 1 | job_embedding) >= threshold_j`

which is the logistic-regression analogue of the cosine-threshold rule:

`cosine(job_embedding, skill_embedding_j) >= threshold_j`


## ELI5: What This Notebook Is Doing

Think of each job as a bundle of numbers called an embedding. We want to use those numbers to guess which skills belong to that job.

### What is OVR?
OVR stands for one-vs-rest.

That means we train one yes/no model for each skill:
- `Accounting Standards`: yes or no?
- `Financial Reporting`: yes or no?
- `Tax Compliance`: yes or no?

So if there are many skills, we train many small binary models instead of one big model.

### How our data fits in
- The CSV gives the true answers: which skills each job already has.
- The train embedding file gives the input features for training.
- The validation embedding file gives the input features for testing how well the model generalizes.
- The skill embeddings file gives the full ACC skill list we are predicting over.

### What the workflow looks like
1. Read jobs and their true skill labels.
2. Turn those labels into a binary matrix.
   Example: if a job has two skills, those two columns become 1 and all others become 0.
3. Train one logistic regression model per skill.
4. Each model outputs a probability, like `0.82` for "this job has Financial Reporting".
5. Apply a threshold to turn that probability into a final yes/no prediction.
6. Compare predicted skills against actual skills using precision, recall, and F1.

### Why thresholding matters
A probability is not yet a final label. We still need to decide what counts as high enough.

For example:
- if threshold = `0.50`, then `0.82` becomes yes
- if threshold = `0.90`, then `0.82` becomes no

This notebook is specifically for the threshold version of OVR, so the core rule is:

`predict skill j if P(skill_j = 1 | job_embedding) >= threshold_j`

### What outputs we want
Best case, we want three main outputs:
- an overall metrics table with strong micro-F1 and decent macro-F1
- a threshold table showing the cutoff used for each skill
- a predictions table showing actual skills vs predicted skills for each job

### What success looks like
The ideal outcome is that this OVR-threshold model performs as well as or better than the cosine-similarity-threshold model on the same validation setup.

In simple terms, best case means:
- better micro-F1
- fewer false positives
- more sensible per-skill decision boundaries


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support

# Data paths
JOB_LABEL_CSV = Path("../data/acc/audit_tax_accounting_jobs.csv")
SKILL_EMBEDDINGS_JSONL = Path("../embedding/acc_skills_embeddings.jsonl")
TRAIN_JOB_EMBEDDINGS_JSONL = Path("../embedding/jobs_train_embeddings.jsonl")
VAL_JOB_EMBEDDINGS_JSONL = Path("../embedding/jobs_val_embeddings.jsonl")
